# 07 External tests
(a) HCP 7T diffusion of the test subjects: another scanner, resolution and set of b-values. (b) HCP test-retest. (c) NSD: other people, another site and protocol, both diffusion runs.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import copy
from retinopath.train import load_trained, net_fodf, trained_runs
subjects = PL.make_subject_lists(cfg); runs = [r for r in trained_runs(cfg) if r[0] in ("path", "voxel", "mlp", "fodrefine")]
s7 = [PL.session(s, "7T") for s in subjects["t7"]]
with Timer(cfg, "07_hcp7t"):
    for v, s in runs:
        c = copy.deepcopy(cfg); model, _ = load_trained(c, v, s, device)
        for ses in s7: net_fodf(c, ses, "NATIVE", f"net_{v}_s{s}", model, device)
    d7 = PL.run_grid(cfg, s7, [("NATIVE", "csd")] + [("NATIVE", f"net_{v}_s{s}") for v, s in runs], "external_7t", ref=None)
d7[["sid", "method", "n_pairs", "n_valid", "rsi_ree_all", "atlas_rsi_ree_all"]].round(3)

In [ ]:
# Test-retest of the reference pipeline (second visit, own anatomy, same vertices)
sr = [PL.session(s, "retest") for s in subjects["retest"]]
with Timer(cfg, "07_retest"):
    dr = PL.run_grid(cfg, sr, [("REF", "msmt_csd")], "retest_rsi", ref=None)
    rr = pd.DataFrame([PL.retest_scores(cfg, s) for s in subjects["retest"]]); rr.to_csv(RES / "retest_overlap.csv", index=False)
rr.round(3)

In [ ]:
# NSD: 8 other people, another scanner and protocol (1.5 mm, b = 1500 and 3000), their own measured pRF maps and
# hand-drawn V1-V3. Nothing here was simulated or seen in training. Downloads about 4 GB per subject if missing.
sn = [PL.session(s, "nsd") for s in subjects.get("nsd", [])]
with Timer(cfg, "07_nsd"):
    for ses in sn: PL.prepare(cfg, ses)
    for v, s in runs:
        c = copy.deepcopy(cfg); model, _ = load_trained(c, v, s, device)
        for ses in sn: net_fodf(c, ses, "NATIVE", f"net_{v}_s{s}", model, device)
    dn = PL.run_grid(cfg, sn, [("NATIVE", "csd"), ("NATIVE", "csd+prob"), ("NATIVE", "proximity")] + [("NATIVE", f"net_{v}_s{s}") for v, s in runs], "external_nsd", ref=None) if sn else None
dn[["sid", "method", "n_pairs", "n_valid", "ree_median_all", "rsi_ree_all", "rsi_ree_geodesic_all", "template_rsi_ree_all", "atlas_rsi_ree_all"]].round(3) if sn else print("NSD: NOT RUN (n_nsd is 0)")

In [ ]:
# NSD run-to-run reproducibility: each subject has a second diffusion run in the same space
if sn and cfg.nsd.use_run2:
    s2 = [PL.session(s, "nsd2") for s in subjects["nsd"]]
    with Timer(cfg, "07_nsd_run2"):
        for ses in s2: PL.prepare(cfg, ses)
        d2 = PL.run_grid(cfg, s2, [("NATIVE", "csd")], "external_nsd_run2", ref=None)
        nr = pd.DataFrame([PL.retest_scores(cfg, s, "NATIVE", "csd", kinds=("nsd", "nsd2")) for s in subjects["nsd"]])
    nr["rsi_run1"] = dn[dn.method == "csd"].set_index("sid").rsi_ree_all.reindex(nr.sid).to_numpy(); nr["rsi_run2"] = d2.set_index("sid").rsi_ree_all.reindex(nr.sid).to_numpy()
    nr.to_csv(RES / "nsd_run_overlap.csv", index=False); display(nr.round(3))
else:
    print("NSD second run: NOT RUN")